In [ ]:
!pip install ipympl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from functools import partial

### What is a wave velocity?

The question may seem trivial, but a few subtleties must be addressed:
- The wave velocity is *not* the material velocity, i.e. not the velocity at which a bit of the fluid travels. Several physical illustrations of this are straightaway evident: scream at a cloud of smoke and the sound (wave) you make will travel faster than the smoke; likewise, a water wave carrying a bit of seaweed will travel far, leaving the weed to bob around more or less in place.
- Instead, we are interested in the speed at which the *wave form* moves through the water. But what does that mean? As we have seen, "the" wave equation can be factored into two first-order equations; this means that any solution is a sum of two *translating* functions $f_\pm(x,t)$ such that $f_\pm(x\mp ct,0)=f_\pm(x, t)$. But translation is a very special symmetry, and more general equations are not guaranteed to respect it! A more general wave may change its shape as it propagates, growing longer or shorter; this is wave dispersion, and we shall see why it bears that name. Nevertheless, we can attempt to treat such a wave as translating and dispersing, and can calculate a "best-fit" translation velocity. 
- Finally, a wave is a sequence of travelling crests and troughs. If a waveform is buckling and changing shape, one can still discern individual crests in it, and track their progression through space. Let us begin my doing just that, and tracking the crests of a sinusoidal wave.


Consider a sinuisoidal wave translating to the right at a fixed velocity $c$; for example, as a solution
to "the" wave equation (which, remember, we have just concluded is nondispersive, with a fixed wave shape). Notice that we can write this as $\cos~k(x-ct),$ where $k=2\pi/\lambda.$ Alternatively, we can expand the paranthesis to yield $$\cos(kx-kct)=:\cos(kx-\omega t) = \mathrm{Re}[\exp~i(kx-\omega t)]$$

a particularly useful form which indicates that at any fixed point $x$, we are left with a *simple harmonic motion* of time period $2\pi/\omega,$ with an initial phase that depends on $x.$ The complex exponential also recalls the idea that an SHM is but the shadow (in red) of a rotating complex vector (in blue).

Throughout this module, you will find the convention that Fourier coefficients of a field (say $\eta$) are denoted with a hat ($\hat\eta$) in math mode, and with a leading underscore (`_eta`) in code.

In [ ]:
"""Setting up the domain"""
L = 10 # length of the domain
xs = np.arange(0,L, L/256) # x in [0,L), 256 steps
t_stop = 100
delta_t = 0.1
ts = np.arange(0, t_stop, delta_t)
c = 1

In [ ]:
%matplotlib widget
"""
This cell is to set up the code for an animation. It requires no action from you.
`phasor_diagram()` draws a phasor diagram illustrating the rotating phasor(s) and its real component.
`update()` evolves the frame of the animation
`make_animation()` creates an HTML object which serves as a movie
"""
def phasor_diagram(wave_list, ax):
    total = 0+0j
    harmonics = []
    for wave in wave_list:
        harmonics.append(
            ax.arrow(0,0, np.real(wave), np.imag(wave),color="black", length_includes_head=True, head_width=0.05, head_length=0.1)
        )
        total+=wave
    signal = ax.arrow(0,0, np.real(total), np.imag(total),color="blue", length_includes_head=True, head_width=0.05, head_length=0.1)
    shadow = ax.arrow(0,0, np.real(total), 0, color="red",length_includes_head=True, head_width=0.05, head_length=0.1)
    max_length = np.sum([np.abs(wave) for wave in wave_list])
    ax.set(xlim=(-max_length-0.1,max_length+0.1),ylim=(-max_length-0.1,max_length+0.1), title=r"SHM at $x=0$ as the red arrow")
    return harmonics, signal, shadow


def update(frame, ax1, wave_list, omegas, line):
    if frame==0:
        return
    ax1.cla()
    _total = 0.+0.j
    for i in range(len(wave_list)):
        wave_list[i] *= np.exp(-1j*omegas[i]*delta_t)
        _total += wave_list[i]
    harmonics, signal, shadow = phasor_diagram([_eta[0] for _eta in wave_list], ax1)
    line.set_ydata(np.real(_total))
    return (harmonics, signal, shadow, line)


def make_animation(wave_list, omegas):
    fig = plt.figure()
    ax1 = fig.add_subplot(121, aspect="equal")
    ax1.set(xlim=(-1.1,1.1),ylim=(-1.1,1.1))
    harmonics, signal, shadow = phasor_diagram([_eta[0] for _eta in wave_list], ax1)
    _total = np.sum(wave_list, axis=0)
    ax2 = fig.add_subplot(122)
    ax2.set(xlabel=r"$x$")
    line  = ax2.plot(xs, np.real(_total))[0]
    plt.tight_layout()
    anim = animation.FuncAnimation(fig=fig, func=partial(update,ax1=ax1, wave_list=wave_list, omegas=omegas, line=line), frames=40, interval=150)
    return anim


In [ ]:
"""We pick a particular sinusoid (specifying its wavenumber and amplitude)
and let it propagate, making an animation"""

ks =  2*np.pi/L  * np.array([5]) # 5th harmonic
omegas = ks*c
wave_list = []
for k in ks:
    wave_list.append(np.exp(1j*k*xs))
coeffs = np.array([1])
anim = make_animation(wave_list, omegas)
display(HTML(anim.to_jshtml()))
plt.close()

This leads to a very important realisation. Suppose that at $t=0,$ we instead have a waveform composed of *two* sinuisoids $\cos k_1x + \cos k_2x.$ Now if we require as before that $f(x,t)=f(x-ct,0),$ then obviously at some other time $t$ the wave must be described by $\cos k_1(x-ct)+\cos k_2(x-ct).$ 
Let us visualise such a composite waveform.

In [ ]:
ks =  2*np.pi/L  * np.array([8, 3]) # 5th harmonic
omegas = ks*c
wave_list = []
coeffs = [1,1]
for a, k in zip(coeffs, ks):
    wave_list.append(a*np.exp(1j*k*xs))
anim = make_animation(wave_list, omegas)
display(HTML(anim.to_jshtml()))
plt.close()

### Exercise 1:

#### 1.1 
Note that any one point (we have visualised $x=0$, without loss of generality) now follows the sum of *two* simple harmonic motions, reflecting the passage of two waves. The two black arrows are the independent SHMs (represented in $\mathbb C$). You can vary the speed of the simulation by changing `c` and `t_stop.` Play around with varying the coefficients of each of these, which we call *Fourier coefficients*, and see how the waveform on the right panel changes in shape. Note that the initial phase of the black arrows can change if we allow the coefficients to be *complex*. What happens when you change the amplitudes? What happens when you change the phases?

#### 1.2
We observe that the superposition looks like wave crests inside an "envelope", because the individual components line up at a period given by the least common multiple of their individual periods. In what follows, we adopt the convention $X(t):=x-ct$ for brevity. Formally, we may write the waveform thus:$$\eta = \mathrm{Re}\left[a_1e^{ik_1X}+a_2e^{ik_2X}\right]$$ Now, pay attention. Let us rewrite $k_1, k_2$ as $k_0\pm k'$ respectively. The foregoing expression becomes $$\mathrm{Re}\left\{~e^{ik_0X}\left[a_1e^{ik'X}+a_2e^{-ik'X}\right]\right\}$$ Expanding this expression (use $ae^{i\kappa x} = (a_r+ia_i)(\cos \kappa x + i\sin\kappa x)$, then take the real part of the final expression) show that this is indeed the product of a sinuisoid and an envelope with wavenumbers $k_0=\frac 1 2 (k_1+k_2)$ and $k' = \frac 1 2 (k_1-k_2).$ Argue that the envelope has a period of $2\pi/k',$ and see if you can explain your observations from part one of this exercise by probing the dependence of the "envelope" factor on the complex coefficients $a_1, a_2$. Make intelligent choices of $k_1, k_2$ from your derivation so that you can clearly see both the envelope and the individual crests inside it visually.

#### 1.3
In the previous exercise, we considered solutions to "the" wave equation, made up of waves travelling at a common speed *c.* Now suppose the two sinusoids travel at distinct speeds $c_1,~c_2$. In other words the waveform becomes $$\mathrm{Re}\left[a_1e^{i(k_1x-\omega_1t)}+a_2e^{i(k_2x-\omega_2t)}\right]$$Factor this out into an envelope and a travelling wave with *different time dependences*. What is the expression you get for the propagation speed of the envelope? Verify this expression by directly modifying `ks` and `omegas` in the code. In particular, (a) make a simulation where the crests and the envelope propagate in *opposite* directions and (b) make a simulation where the envelope stays fixed in place whie the crests propagate. Note that the speed of the crests is called *phase speed* and denoted $c_p$ or $c^{~p}$, while that of the envelope is called *group speed* and denoted $c_g$ or $c^{~g}$.